# PaddleOCR 파인튜닝 가이드

운송장 이미지 OCR을 위한 PaddleOCR 모델 파인튜닝

## 목차
1. 환경 설정
2. 데이터 준비 (PaddleOCR 형식 변환)
3. 설정 파일 생성
4. 모델 학습
5. 모델 평가 및 추론

## 1. 환경 설정

In [1]:
# PaddlePaddle 및 PaddleOCR 설치
# GPU 버전 (CUDA 11.x)
!pip install paddlepaddle-gpu -i https://pypi.tuna.tsinghua.edu.cn/simple

# CPU 버전 (GPU가 없는 경우)
# !pip install paddlepaddle -i https://pypi.tuna.tsinghua.edu.cn/simple

# PaddleOCR 설치
!pip install paddleocr

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


ERROR: Could not find a version that satisfies the requirement paddlepaddle-gpu (from versions: none)
ERROR: No matching distribution found for paddlepaddle-gpu


  Using cached paddleocr-3.3.3-py3-none-any.whl.metadata (55 kB)
  Using cached paddlex-3.3.13-py3-none-any.whl.metadata (79 kB)
  Using cached aistudio_sdk-0.3.8-py3-none-any.whl.metadata (1.1 kB)
  Using cached chardet-5.2.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached colorlog-6.10.1-py3-none-any.whl.metadata (11 kB)
  Using cached huggingface_hub-1.3.4-py3-none-any.whl.metadata (13 kB)
  Using cached modelscope-1.34.0-py3-none-any.whl.metadata (43 kB)
  Using cached prettytable-3.17.0-py3-none-any.whl.metadata (34 kB)
  Using cached py_cpuinfo-9.0.0-py3-none-any.whl.metadata (794 bytes)
  Using cached imagesize-1.4.1-py2.py3-none-any.whl.metadata (1.5 kB)
  Using cached opencv_contrib_python-4.10.0.84-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached pypdfium2-5.3.0-py3-none-win_amd64.whl.metadata (67 kB)
  Using cached bce_python_sdk-0.9.60-py3-none-any.whl.metadata (416 bytes)
  Using cached pycryptodome-3.23.0-cp37-abi3-win_amd64.whl.metadata (3.5 kB)
  Using cached f

  You can safely remove it manually.


In [ ]:
# PaddleOCR 소스 클론 (파인튜닝에 필요)
!git clone https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR
!pip install -r requirements.txt

In [ ]:
import os
import json
import shutil
from pathlib import Path
from PIL import Image
import random

# 경로 설정
BASE_DIR = Path('../')  # ai 폴더
GENERATED_DIR = BASE_DIR / 'generated'
PADDLE_DATA_DIR = BASE_DIR / 'paddle_data'

print(f"생성된 이미지 폴더: {GENERATED_DIR}")
print(f"PaddleOCR 데이터 폴더: {PADDLE_DATA_DIR}")

## 2. 데이터 준비 - PaddleOCR 형식 변환

PaddleOCR은 두 가지 태스크를 위한 데이터 형식이 있습니다:
- **텍스트 감지 (Detection)**: 이미지에서 텍스트 영역 찾기
- **텍스트 인식 (Recognition)**: 크롭된 텍스트 이미지를 문자로 변환

우리 데이터는 필드별 바운딩 박스가 있으므로 **텍스트 인식** 모델을 파인튜닝합니다.

In [ ]:
def convert_to_paddle_recognition_format(generated_dir: Path, output_dir: Path, train_ratio: float = 0.9):
    """
    생성된 데이터를 PaddleOCR 텍스트 인식 형식으로 변환
    
    PaddleOCR Recognition 형식:
    - 이미지: 텍스트 영역만 크롭된 이미지
    - 라벨: image_path\tlabel 형식의 txt 파일
    """
    
    # 출력 디렉토리 생성
    train_img_dir = output_dir / 'train' / 'images'
    val_img_dir = output_dir / 'val' / 'images'
    train_img_dir.mkdir(parents=True, exist_ok=True)
    val_img_dir.mkdir(parents=True, exist_ok=True)
    
    # 라벨 파일 로드
    labels_file = generated_dir / 'labels' / 'labels.json'
    with open(labels_file, 'r', encoding='utf-8') as f:
        all_labels = json.load(f)
    
    # 데이터 섞기
    random.shuffle(all_labels)
    
    # Train/Val 분할
    split_idx = int(len(all_labels) * train_ratio)
    train_labels = all_labels[:split_idx]
    val_labels = all_labels[split_idx:]
    
    train_records = []
    val_records = []
    
    def process_labels(labels, img_dir, records, prefix):
        for idx, label_data in enumerate(labels):
            # 원본 이미지 로드
            img_path = generated_dir / label_data['image_path']
            if not img_path.exists():
                continue
            
            img = Image.open(img_path)
            
            # 각 필드별로 크롭하여 저장
            for field_idx, field in enumerate(label_data['fields']):
                bbox = field['bbox']  # [x1, y1, x2, y2]
                text = field['text']
                field_name = field['field_name']
                
                # 이미지 크롭
                cropped = img.crop((bbox[0], bbox[1], bbox[2], bbox[3]))
                
                # 파일명 생성
                crop_filename = f"{prefix}_{idx:05d}_{field_name}.jpg"
                crop_path = img_dir / crop_filename
                
                # 저장
                cropped.save(crop_path, 'JPEG', quality=95)
                
                # 레코드 추가 (상대 경로 사용)
                relative_path = f"images/{crop_filename}"
                records.append(f"{relative_path}\t{text}")
            
            if (idx + 1) % 100 == 0:
                print(f"{prefix}: {idx + 1}/{len(labels)} 처리 완료")
    
    print("Train 데이터 처리 중...")
    process_labels(train_labels, train_img_dir, train_records, 'train')
    
    print("\nVal 데이터 처리 중...")
    process_labels(val_labels, val_img_dir, val_records, 'val')
    
    # 라벨 파일 저장
    with open(output_dir / 'train' / 'label.txt', 'w', encoding='utf-8') as f:
        f.write('\n'.join(train_records))
    
    with open(output_dir / 'val' / 'label.txt', 'w', encoding='utf-8') as f:
        f.write('\n'.join(val_records))
    
    print(f"\n변환 완료!")
    print(f"Train 샘플 수: {len(train_records)}")
    print(f"Val 샘플 수: {len(val_records)}")
    
    return len(train_records), len(val_records)

In [ ]:
# 데이터 변환 실행
train_count, val_count = convert_to_paddle_recognition_format(
    GENERATED_DIR, 
    PADDLE_DATA_DIR,
    train_ratio=0.9
)

print(f"\nTrain 이미지: {train_count}개")
print(f"Val 이미지: {val_count}개")

In [ ]:
# 변환된 데이터 확인
print("=" * 50)
print("Train 라벨 샘플:")
print("=" * 50)
with open(PADDLE_DATA_DIR / 'train' / 'label.txt', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 10:
            break
        print(line.strip())

## 3. 한글 문자 사전 생성

PaddleOCR은 인식할 문자 목록이 필요합니다.

In [ ]:
def create_korean_dict(data_dir: Path, output_file: Path):
    """
    학습 데이터에서 사용된 모든 문자를 추출하여 사전 생성
    """
    chars = set()
    
    # Train 라벨에서 문자 추출
    label_file = data_dir / 'train' / 'label.txt'
    with open(label_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                text = parts[1]
                chars.update(text)
    
    # 기본 한글 완성형 추가 (가-힣)
    for code in range(0xAC00, 0xD7A4):
        chars.add(chr(code))
    
    # 숫자, 영문, 특수문자 추가
    chars.update('0123456789')
    chars.update('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ')
    chars.update(' .-_,()[]{}:;/\\@#$%&*+=<>?!"\'')
    
    # 정렬 후 저장
    sorted_chars = sorted(chars)
    
    with open(output_file, 'w', encoding='utf-8') as f:
        for char in sorted_chars:
            f.write(char + '\n')
    
    print(f"문자 사전 생성 완료: {len(sorted_chars)}개 문자")
    print(f"저장 위치: {output_file}")
    
    return sorted_chars

In [ ]:
# 문자 사전 생성
dict_file = PADDLE_DATA_DIR / 'korean_dict.txt'
chars = create_korean_dict(PADDLE_DATA_DIR, dict_file)

## 4. 설정 파일 생성

PaddleOCR 학습을 위한 YAML 설정 파일을 생성합니다.

In [ ]:
# 설정 파일 생성
config_content = f'''
Global:
  debug: false
  use_gpu: true
  epoch_num: 100
  log_smooth_window: 20
  print_batch_step: 10
  save_model_dir: ./output/rec_korean_finetune
  save_epoch_step: 10
  eval_batch_step: [0, 500]
  cal_metric_during_train: true
  pretrained_model: ./pretrain_models/korean_PP-OCRv3_rec_train/best_accuracy
  checkpoints:
  save_inference_dir:
  use_visualdl: false
  infer_img: 
  character_dict_path: {str(PADDLE_DATA_DIR / 'korean_dict.txt').replace(chr(92), '/')}
  max_text_length: 50
  infer_mode: false
  use_space_char: true
  distributed: true
  save_res_path: ./output/rec/predicts.txt

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.0005
    warmup_epoch: 5
  regularizer:
    name: L2
    factor: 3.0e-05

Architecture:
  model_type: rec
  algorithm: SVTR_LCNet
  Transform:
  Backbone:
    name: MobileNetV1Enhance
    scale: 0.5
    last_conv_stride: [1, 2]
    last_pool_type: avg
  Head:
    name: MultiHead
    head_list:
      - CTCHead:
          Neck:
            name: svtr
            dims: 64
            depth: 2
            hidden_dims: 120
            use_guide: True
          Head:
            fc_decay: 0.00001
      - SARHead:
          enc_dim: 512
          max_text_length: 50

Loss:
  name: MultiLoss
  loss_config_list:
    - CTCLoss:
    - SARLoss:

PostProcess:  
  name: CTCLabelDecode

Metric:
  name: RecMetric
  main_indicator: acc
  ignore_space: False

Train:
  dataset:
    name: SimpleDataSet
    data_dir: {str(PADDLE_DATA_DIR / 'train').replace(chr(92), '/')}
    ext_op_transform_idx: 1
    label_file_list:
      - {str(PADDLE_DATA_DIR / 'train' / 'label.txt').replace(chr(92), '/')}
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - RecConAug:
          prob: 0.5
          ext_data_num: 2
          image_shape: [48, 320, 3]
      - RecAug:
      - MultiLabelEncode:
      - RecResizeImg:
          image_shape: [3, 48, 320]
      - KeepKeys:
          keep_keys:
            - image
            - label_ctc
            - label_sar
            - length
            - valid_ratio
  loader:
    shuffle: true
    batch_size_per_card: 64
    drop_last: true
    num_workers: 4

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: {str(PADDLE_DATA_DIR / 'val').replace(chr(92), '/')}
    label_file_list:
      - {str(PADDLE_DATA_DIR / 'val' / 'label.txt').replace(chr(92), '/')}
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - MultiLabelEncode:
      - RecResizeImg:
          image_shape: [3, 48, 320]
      - KeepKeys:
          keep_keys:
            - image
            - label_ctc
            - label_sar
            - length
            - valid_ratio
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: 64
    num_workers: 4
'''

config_file = PADDLE_DATA_DIR / 'rec_korean_finetune.yml'
with open(config_file, 'w', encoding='utf-8') as f:
    f.write(config_content)

print(f"설정 파일 저장: {config_file}")

## 5. 사전 학습 모델 다운로드

In [ ]:
# 한국어 PP-OCRv3 사전 학습 모델 다운로드
import urllib.request
import tarfile

pretrain_dir = Path('pretrain_models')
pretrain_dir.mkdir(exist_ok=True)

# 한국어 인식 모델 다운로드 (PP-OCRv3)
model_url = "https://paddleocr.bj.bcebos.com/PP-OCRv3/multilingual/korean_PP-OCRv3_rec_train.tar"
model_tar = pretrain_dir / "korean_PP-OCRv3_rec_train.tar"

if not (pretrain_dir / "korean_PP-OCRv3_rec_train").exists():
    print("사전 학습 모델 다운로드 중...")
    urllib.request.urlretrieve(model_url, model_tar)
    
    print("압축 해제 중...")
    with tarfile.open(model_tar, 'r') as tar:
        tar.extractall(pretrain_dir)
    
    # tar 파일 삭제
    model_tar.unlink()
    print("완료!")
else:
    print("사전 학습 모델이 이미 존재합니다.")

## 6. 모델 학습

In [ ]:
# 학습 실행 (GPU 사용)
!python tools/train.py -c {str(config_file).replace(chr(92), '/')}

In [ ]:
# CPU로 학습 (GPU가 없는 경우)
# !python tools/train.py -c {str(config_file).replace(chr(92), '/')} -o Global.use_gpu=false

## 7. 모델 평가

In [ ]:
# 학습된 모델 평가
!python tools/eval.py -c {str(config_file).replace(chr(92), '/')} \
    -o Global.checkpoints=./output/rec_korean_finetune/best_accuracy

## 8. 추론 모델 내보내기

In [ ]:
# 추론용 모델로 변환
!python tools/export_model.py -c {str(config_file).replace(chr(92), '/')} \
    -o Global.pretrained_model=./output/rec_korean_finetune/best_accuracy \
    Global.save_inference_dir=./output/rec_korean_finetune/inference

## 9. 파인튜닝된 모델로 추론 테스트

In [ ]:
from paddleocr import PaddleOCR

# 파인튜닝된 모델 로드
ocr = PaddleOCR(
    rec_model_dir='./output/rec_korean_finetune/inference',
    rec_char_dict_path=str(dict_file),
    use_angle_cls=False,
    lang='korean'
)

# 테스트 이미지로 추론
test_image = str(BASE_DIR / 'img' / '운송장 예시파일.jpg')
result = ocr.ocr(test_image, cls=False)

print("OCR 결과:")
print("=" * 50)
for line in result[0]:
    bbox, (text, confidence) = line
    print(f"텍스트: {text}")
    print(f"신뢰도: {confidence:.4f}")
    print("-" * 30)

In [ ]:
# 생성된 이미지로 테스트
import matplotlib.pyplot as plt
from PIL import Image

test_images = list((GENERATED_DIR / 'images').glob('*.jpg'))[:5]

fig, axes = plt.subplots(1, len(test_images), figsize=(20, 4))

for i, img_path in enumerate(test_images):
    result = ocr.ocr(str(img_path), cls=False)
    
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].axis('off')
    
    # 인식된 텍스트 표시
    texts = [line[1][0] for line in result[0]] if result[0] else []
    axes[i].set_title('\n'.join(texts[:3]), fontsize=8)

plt.tight_layout()
plt.show()

## 10. 모델 저장 및 배포

In [ ]:
# 최종 모델을 ai 폴더로 복사
import shutil

final_model_dir = BASE_DIR / 'models' / 'paddleocr_korean_finetuned'
final_model_dir.mkdir(parents=True, exist_ok=True)

# 추론 모델 복사
inference_dir = Path('./output/rec_korean_finetune/inference')
if inference_dir.exists():
    for file in inference_dir.glob('*'):
        shutil.copy(file, final_model_dir)
    
    # 문자 사전도 복사
    shutil.copy(dict_file, final_model_dir / 'korean_dict.txt')
    
    print(f"모델 저장 완료: {final_model_dir}")
else:
    print("추론 모델을 먼저 내보내세요.")

## 완료!

파인튜닝된 모델 사용법:

```python
from paddleocr import PaddleOCR

ocr = PaddleOCR(
    rec_model_dir='models/paddleocr_korean_finetuned',
    rec_char_dict_path='models/paddleocr_korean_finetuned/korean_dict.txt',
    use_angle_cls=False,
    lang='korean'
)

result = ocr.ocr('your_image.jpg', cls=False)
```